Author: Andreas Loizias Created: 2026-03-17 Description: This is part 3 of the card fraud detection ML model. Here we import the model, features, scaler, and test data and make predictions and evaluate it.

In [1]:
#load importan libraries
import pandas as pd
import numpy as np
import joblib
from tensorflow.keras.models import load_model

2026-03-17 07:23:09.769705: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773732189.998749      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773732190.070524      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773732190.588024      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773732190.588091      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773732190.588097      55 computation_placer.cc:177] computation placer alr

In [5]:
#load model, features list, scaler
model = load_model("/kaggle/input/datasets/andreasloizias/frad-detection-artifacts/fraud_nn_model.keras")
scaler = joblib.load("/kaggle/input/datasets/andreasloizias/frad-detection-artifacts/scaler.pkl")
feature_names = joblib.load("/kaggle/input/datasets/andreasloizias/frad-detection-artifacts/feature_names.pkl")

In [6]:
#load test data features and targets
x_test = np.load("/kaggle/input/datasets/andreasloizias/frad-detection-artifacts/X_test.npy")
y_test = np.load("/kaggle/input/datasets/andreasloizias/frad-detection-artifacts/y_test.npy")

In [7]:
#make sure they loaded
print(x_test.shape)
print(y_test.shape)

(56962, 30)
(56962,)


In [8]:
# check if values are scaled
print(x_test[:5])
#they are scaled so no need to scale them using the scaler

[[-0.79084792 -0.39651134  0.67932606  0.94415138  0.21419284  0.33862802
  -0.07431404  0.59499291  0.02038216 -0.65695689 -0.08330618 -0.42584836
  -0.22028157  0.0362617   0.34606134  1.8097152  -0.50530793  0.20020771
  -1.0267568   1.30163294  0.2841844  -0.65931264 -1.86841071 -0.04846475
  -0.77323237 -0.14064185  0.24363799  0.38044107  0.39965732 -0.27224598]
 [ 0.5052358   0.98591857 -1.54387505  0.53183878 -0.68828866 -2.12068775
   0.4905842  -2.07645522  0.43675111  0.54594249  1.24512674 -0.45167898
  -0.54379691 -1.58914506 -1.35805105 -2.6759359  -0.62409176  1.04041341
   1.11510204  0.44616125 -0.49607668  0.02726303  0.93753044  0.22847383
   0.10956105 -0.93997709 -0.01231079  0.21268644 -0.07631249  0.06304997]
 [-0.32810533  0.61609379 -0.31474302  0.5581927  -0.53696149 -0.8494635
  -0.29656065 -0.61573071  0.17230888  1.89149208 -0.98844489 -0.49686163
  -0.08891743 -1.79245805  0.2322034   1.87265581 -0.74823802  0.27494993
  -0.05746495  0.25432559 -0.34273029

In [11]:
# make predictions on test data
y_pred_prob = model.predict(x_test).ravel()

threshold = 0.3
y_pred = (y_pred_prob >= threshold).astype(int)

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [12]:
# evaluate predictions
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[54495  2369]
 [    8    90]]
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     56864
           1       0.04      0.92      0.07        98

    accuracy                           0.96     56962
   macro avg       0.52      0.94      0.52     56962
weighted avg       1.00      0.96      0.98     56962



We see from the evaluation that we predicted 90 true positives but also 2369 false positives. Our precision is low but recall is high which explains why we only missed 8 fraud cases and captured 90 but mislabeled 2369 legit transactions as fraud.

In [20]:
#create a predict transactions function
def predict_transactions(X, model, scaler=None, threshold=0.3, already_scaled=False):
    """
    Predict fraud probabilities and classes for a batch of transactions.

    Parameters:
        X : array-like
            Input features.
        model : trained keras model
        scaler : fitted scaler
        threshold : float
            Classification threshold.
        already_scaled : bool
            If True, skip scaling.

    Returns:
        probas : np.ndarray
            Fraud probabilities.
        preds : np.ndarray
            Predicted classes (0 or 1).
    """
    X_input = X.copy()

    if not already_scaled:
        if scaler is None:
            raise ValueError("Scaler is required when already_scaled=False")
        X_input = scaler.transform(X_input)

    probas = model.predict(X_input).ravel()
    preds = (probas >= threshold).astype(int)

    return probas, preds

In [22]:
#call function to make predictions
y_prob, y_pred = predict_transactions(
    x_test,
    model=model,
    scaler=scaler,
    threshold=0.3,
    already_scaled=True
)

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step


In [23]:
# put results in a data frame
import pandas as pd

results = pd.DataFrame(x_test, columns=feature_names)
results["actual_class"] = y_test
results["fraud_probability"] = y_prob
results["predicted_class"] = y_pred

results.head(10)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V23,V24,V25,V26,V27,V28,Amount,actual_class,fraud_probability,predicted_class
0,-0.790848,-0.396511,0.679326,0.944151,0.214193,0.338628,-0.074314,0.594993,0.020382,-0.656957,...,-0.048465,-0.773232,-0.140642,0.243638,0.380441,0.399657,-0.272246,0,0.014741,0
1,0.505236,0.985919,-1.543875,0.531839,-0.688289,-2.120688,0.490584,-2.076455,0.436751,0.545942,...,0.228474,0.109561,-0.939977,-0.012311,0.212686,-0.076312,0.063050,0,0.178847,0
2,-0.328105,0.616094,-0.314743,0.558193,-0.536961,-0.849464,-0.296561,-0.615731,0.172309,1.891492,...,0.023694,0.082364,0.708164,-1.349709,0.240886,0.083490,-0.347683,0,0.083241,0
3,-1.302422,0.386174,-0.840710,-0.081585,-0.172742,-0.674656,-0.154417,0.024288,-0.117577,-0.844321,...,-0.248190,0.277238,0.295930,2.276015,-0.206551,0.131967,0.790815,0,0.007141,0
4,1.497628,0.955655,0.008748,-1.179025,0.857747,0.525008,-0.283419,0.496608,-0.199710,-0.168518,...,-0.065824,1.208182,0.848792,-1.144762,-0.075560,-0.156772,-0.054433,0,0.021657,0
5,-0.771383,0.517504,-0.254876,0.938142,1.201834,-0.819561,0.432301,-0.642815,0.349138,1.203131,...,-0.040769,0.537226,0.848865,-0.915196,0.172504,0.080789,-0.192879,0,0.099688,0
6,-1.092454,0.606161,-0.490734,0.138993,-0.332701,-0.846220,-0.647351,-0.305013,-0.092312,-0.681705,...,0.098760,0.671075,0.155714,2.164793,-0.177883,0.045447,-0.000674,0,0.022697,0
7,-0.829863,-0.116264,0.636591,1.058179,1.422350,0.592211,0.240553,0.726441,-0.168443,-1.100173,...,0.067837,-1.096213,-1.210710,-0.632125,-0.040228,-0.156635,-0.200661,0,0.045540,0
8,0.857443,0.937812,-0.264848,0.023036,0.868398,-0.515474,0.145366,-0.656172,0.195786,0.904357,...,0.312041,-0.644718,-0.673884,-1.569871,0.157512,-0.083248,-0.172987,0,0.104902,0
9,-0.157552,-0.647371,0.037075,-0.936986,-1.696021,1.015873,-1.393906,0.635505,0.138827,0.100749,...,-0.607556,-0.638625,-0.712875,1.095070,0.286352,0.227342,-0.347683,0,0.013652,0


In [24]:
# create function for making one prediction
def predict_one_transaction(x_row, model, scaler=None, threshold=0.3, already_scaled=False):
    """
    Predict fraud for a single transaction.
    x_row should be shape (n_features,) or (1, n_features)
    """
    x_row = np.array(x_row)

    if x_row.ndim == 1:
        x_row = x_row.reshape(1, -1)

    prob, pred = predict_transactions(
        x_row,
        model=model,
        scaler=scaler,
        threshold=threshold,
        already_scaled=already_scaled
    )

    return prob[0], pred[0]

In [26]:
# call function to make one prediction
single_prob, single_pred = predict_one_transaction(
    x_test[0],
    model=model,
    scaler=scaler,
    threshold=0.3,
    already_scaled=False   # change to True if X_test is already scaled
)

print("Fraud probability:", single_prob)
print("Predicted class:", single_pred)
print("Actual class:", y_test[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
Fraud probability: 0.013393107
Predicted class: 0
Actual class: 0


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


For one row we get probability of fraud 0.0134. It predicts it as non-fraud

In [27]:
# display only the fraud predictions
frauds_found = results[results["predicted_class"] == 1]
frauds_found.head(20)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V23,V24,V25,V26,V27,V28,Amount,actual_class,fraud_probability,predicted_class
11,-1.841825,-0.118060,0.742105,0.523058,0.815479,0.752608,-0.251069,0.824807,-0.309760,0.686961,...,-0.597244,-1.330551,0.446307,-0.492526,-0.198678,-0.312248,-0.254340,0,0.422972,1
17,0.659041,0.066421,0.730265,-0.963863,-0.255343,1.141582,-0.916961,1.375813,-0.343908,-0.181297,...,-0.420264,-1.163335,0.169344,-1.272109,-0.196218,-0.376673,-0.129313,0,0.646867,1
18,1.310580,1.038070,0.327658,-1.641250,0.413267,0.539549,-0.976035,0.218770,-0.212379,0.232844,...,-0.158809,-0.908116,0.526066,-0.209605,0.040840,-0.017220,-0.347683,0,0.393857,1
34,1.617305,1.066453,0.450009,-2.005511,0.565407,0.790343,-1.391233,0.689453,-0.452511,-0.053663,...,-0.001346,1.631831,0.700614,1.354352,-0.190242,-0.025833,-0.348635,0,0.633172,1
48,0.548717,-2.402602,-3.578042,0.208853,1.360066,3.784881,-3.026543,-3.625782,-0.060309,1.869944,...,-2.362442,2.335076,-5.207690,0.817433,2.724156,1.693063,-0.305994,0,0.872744,1
50,-1.502952,0.564352,0.351489,0.353074,2.168986,-0.000622,0.114954,-0.208564,0.103250,1.148336,...,-0.066102,-0.192620,0.920019,0.152755,0.005081,0.096625,-0.321558,0,0.734555,1
77,0.799975,-0.223640,1.183224,-1.536533,-0.483124,1.683946,-0.266507,0.932416,-1.808389,-0.407980,...,-0.330674,-0.562673,0.516553,1.208560,-1.350783,-0.528528,-0.348635,0,0.363305,1
94,0.781942,-0.484029,1.220691,-2.177942,-0.108408,-0.376205,0.257339,0.111669,-4.578189,-1.027797,...,0.621688,-0.508040,-2.211142,-0.876478,1.374194,0.920137,0.708470,0,0.550744,1
102,-1.206233,0.486556,0.211873,0.160857,1.911804,0.197021,0.525200,-0.032332,0.195538,-0.544922,...,-0.302904,-0.589329,1.083709,0.206727,0.063061,0.125321,-0.052646,0,0.501973,1
135,1.095008,-2.008856,1.510951,0.798062,-0.919651,-1.338581,2.376772,-4.326519,-8.522470,1.344861,...,1.142896,0.922720,0.598752,2.875732,0.241239,0.086400,-0.311949,0,0.498284,1
